In [1]:
# manejo de datos
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

# api Xenocanto
from xenopy import Query
from scipy import signal
import scipy.signal.windows

# manejo de audio
import librosa
import librosa.display

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# Ruta al archivo MP3
ruta_al_archivo = 'prueba.mp3'

# Cargar el archivo MP3
y, sr = librosa.load(ruta_al_archivo)

# 'y' es un array de numpy que contiene los datos de audio
# 'sr' es la frecuencia de muestreo del audio (número de muestras por segundo)

In [3]:
# Set the hop length; at 22050 Hz, 512 samples ~= 23ms
hop_length = 512

# Separate harmonics and percussives into two waveforms
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [4]:
# Beat track on the percussive signal
tempo, beat_frames = librosa.beat.beat_track(y=y_percussive, sr=sr)

In [5]:
# Compute MFCC features from the raw signal
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)

In [6]:
# And the first-order differences (delta features)
mfcc_delta = librosa.feature.delta(mfcc)

In [7]:
# Stack and synchronize between beat events
# This time, we'll use the mean value (default) instead of median
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [8]:
# Compute chroma features from the harmonic signal
chromagram = librosa.feature.chroma_cqt(y=y_harmonic, sr=sr)

In [ ]:
'Melspectrogram': 
'Polynomial coefficients': 
'Fourier tempogram': 
'Ceros Cruzados': Es la tasa a la que la señal cambia de signo.
'Perceptrual Sharpness': Es una medida de la nitidez percibida de la señal de audio.
'Loudness': Es la suma de la RMS de la señal de audio.
'Tonnetz': Es una representación de la tonalidad de una señal de audio.
'Contraste': Es la diferencia entre los picos y los valles en un espectro.
'Rolloff': Es la frecuencia por debajo de la cual se encuentra una cierta cantidad de la energía espectral.
'Centroide': Es el centro de gravedad del espectro.
'Bandwidth': Es el ancho de la banda de frecuencias en la que se encuentra la mayor parte de la energía.

In [9]:
# Extraer características de audio
features = {
    'Longitud de onda': len(y), # Es la longitud total de la señal de audio.
    'Intensidad miníma': np.min(librosa.feature.rms(y=y)), # Es el valor mínimo de la raíz cuadrada media (RMS) de la señal de audio.
    'Intensidad media': np.mean(librosa.feature.rms(y=y)), # Es el valor medio de la RMS de la señal de audio.
    'Intensidad máxima': np.max(librosa.feature.rms(y=y)), # Es el valor máximo de la RMS de la señal de audio.
    'Tonos principales': len(librosa.core.piptrack(y=y)[1]), # Es el número de tonos principales en la señal de audio.
    'Melodía': np.mean(librosa.feature.tonnetz(y=y, sr=sr)), # Es la melodía promedio de la señal de audio.
    'Centroide espectral': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)), # Es el centroide espectral promedio de la señal de audio.
    'Rolloff espectral': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)), # Es la frecuencia por debajo de la cual se encuentra una cierta cantidad de la energía espectral.
    'Contraste espectral': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)), # Es la diferencia entre los picos y los valles en un espectro.
    'Ancho de banda espectral': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)), # Es el ancho de la banda de frecuencias en la que se encuentra la mayor parte de la energía.
    'Croma': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)), # Es una representación de la distribución de energía por tonos.
    'Tempo': librosa.beat.tempo(y=y, sr=sr)[0], # Es la velocidad o el ritmo de una pieza musical.
    'Pulso': np.mean(librosa.feature.tempogram(y=y, sr=sr)), # Es una representación del ritmo en una señal de audio.
    'Coeficientes MFCC': np.mean(librosa.feature.mfcc(y=y, sr=sr)), # Son los coeficientes de los cepstrum de frecuencia mel, que son una representación del espectro de potencia de una señal de audio
    'RMS': np.mean(librosa.feature.rms(y=y)), # Es la raíz cuadrada media de la señal de audio.
    'Cens': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)), # Es una versión suavizada del croma.
    'Piptrack': np.mean(librosa.core.piptrack(y=y)[1]), # Es una representación de los picos de una señal de audio.
    'Cruces por cero': np.mean(librosa.feature.zero_crossing_rate(y)), # Es la tasa a la que la señal cambia de signo.
    'Cromagrama CQT': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)), # Es una representación del croma utilizando una transformada de calidad constante.
    'Cromagrama CENS': np.mean(librosa.feature.chroma_cens(y=y, sr=sr)), # Es una versión suavizada del cromagrama CQT.
    'Melspectrogram': np.mean(librosa.feature.melspectrogram(y=y, sr=sr)), # Es una representación del espectro de potencia en una escala de frecuencia mel.
    'Polynomial coefficients': np.mean(librosa.feature.poly_features(y=y, sr=sr)), # Son los coeficientes de un polinomio que se ajusta a la señal de audio.
    'Fourier tempogram': np.mean(librosa.feature.fourier_tempogram(y=y, sr=sr)), # Es una representación del ritmo basada en la transformada de Fourier.
    'Ceros Cruzados': np.mean(librosa.feature.zero_crossing_rate(y)),
    'Perceptrual Sharpness': np.mean(librosa.feature.spectral_flatness(y=y)),
    'Loudness': np.sum(librosa.feature.rms(y=y)),
    'Tonnetz': np.mean(librosa.feature.tonnetz(y=y, sr=sr)),
    'Contraste': np.mean(librosa.feature.spectral_contrast(y=y, sr=sr)),
    'Rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
    'Centroide': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
    'Bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    } 

# Crear un DataFrame para almacenar los datos
df = pd.DataFrame(features, index=[0])

# Guardar los datos en un archivo CSV
#df.to_csv('datos_de_audio.csv', index=False)
df.head()

,Longitud de onda,Intensidad miníma,Intensidad media,Intensidad máxima,Tonos principales,Melodía,Centroide espectral,Rolloff espectral,Contraste espectral,Ancho de banda espectral,Croma,Tempo,Pulso,Coeficientes MFCC,RMS,Cens,Piptrack,Cruces por cero,Cromagrama CQT,Cromagrama CENS,Melspectrogram,Polynomial coefficients,Fourier tempogram,Ceros Cruzados,Perceptrual Sharpness,Loudness,Tonnetz,Contraste,Rolloff,Centroide,Bandwidth
0,197568,0.032679,0.081596,0.194569,1025,0.000042,5639.317557,7371.802833,21.260144,1898.440747,0.337589,129.199219,0.250223,-24.013008,0.081596,0.282174,0.055431,0.541826,0.622486,0.282174,0.356371,0.556976,0.427478-0.009859j,0.541826,0.039559,31.495869,0.000042,21.260144,7371.802833,5639.317557,1898.440747


In [10]:
# Aggregate chroma features between beat events
# We'll use the median value of each feature between beat frames
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [11]:
# Finally, stack all beat-synchronous features together
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])

In [12]:
y_harmonic, y_percussive = librosa.effects.hpss(y)

In [13]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)
mfcc

In [20]:
mfcc_delta = librosa.feature.delta(mfcc)
mfcc_delta

array([[16.855936  , 16.855936  , 16.855936  , ..., -0.89970374,
        -0.89970374, -0.89970374],
       [-3.4494283 , -3.4494283 , -3.4494283 , ...,  1.7929302 ,
         1.7929302 ,  1.7929302 ],
       [-4.0492163 , -4.0492163 , -4.0492163 , ...,  1.7142378 ,
         1.7142378 ,  1.7142378 ],
       ...,
       [-3.6456344 , -3.6456344 , -3.6456344 , ...,  1.7010834 ,
         1.7010834 ,  1.7010834 ],
       [-1.1730183 , -1.1730183 , -1.1730183 , ...,  1.0955445 ,
         1.0955445 ,  1.0955445 ],
       [ 1.2888954 ,  1.2888954 ,  1.2888954 , ...,  0.47902873,
         0.47902873,  0.47902873]], dtype=float32)

In [15]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [16]:
beat_mfcc_delta = librosa.util.sync(np.vstack([mfcc, mfcc_delta]),
                                    beat_frames)

In [17]:
beat_chroma = librosa.util.sync(chromagram,
                                beat_frames,
                                aggregate=np.median)

In [18]:
beat_features = np.vstack([beat_chroma, beat_mfcc_delta])